# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rana4682/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
import subprocess
import sys
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Rana4682/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
            check=True
        )
    os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(df.shape)

df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 1. Question

*# Research Question

Can search performance signals be used to identify content pages that should be prioritized for refresh?

This project supports editorial decision-making by ranking content pages according to their refresh priority.

Unit of Analysis:
One content page.

Output:
A ranked refresh priority score.

Decision:
Editors can use this ranking to prioritize content updates.

The recommendations are intended for decision-support and do not guarantee ranking improvements.*

In [2]:
print("Research Question")
print("-" * 50)
print("Can search performance signals identify pages that should be refreshed?")
print("Unit of Analysis : One content page")
print("Output           : Ranked Refresh Score")
print("Decision         : Content Refresh Priority")

Research Question
--------------------------------------------------
Can search performance signals identify pages that should be refreshed?
Unit of Analysis : One content page
Output           : Ranked Refresh Score
Decision         : Content Refresh Priority


## 2. Data

*# Data

Dataset: FlyRank Internship Warehouse Dataset

Table Used:
content_refresh_anonymized.csv

Features Used:
- ctr
- avg_position
- trend_pct

Excluded Columns:
- content_id
- client_id

Reason:
Identifier fields were excluded because they are not predictive features. Only anonymized and public-safe data was used..*

In [3]:
print("Dataset Shape")
print(df.shape)

print("\nDataset Columns")
print(df.columns.tolist())

print("\nMissing Values")
print(df.isnull().sum())

Dataset Shape
(30000, 44)

Dataset Columns
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

Missing Values
content_id                    0
client_id                     0
search_volume              2468
competition                2468
competition_level          2610
cpc       

## 3. Methodology

# Methodology

This project follows the Refresh / Content Opportunity Scoring lane.

Baseline:
A rule-based baseline score was created using CTR, Average Position, and Trend Percentage.

Model:
Random Forest Regressor.

Features Used:
- ctr
- avg_position
- trend_pct

Target:
Baseline Refresh Score.

Validation:
A train/test split was used to compare the machine learning model against the baseline.

Leakage Check:
No client identifiers or future-window information were used as model features. The model uses only observed search performance signals available at prediction time.

The results are intended for decision-support only..*

In [5]:
from sklearn.model_selection import train_test_split

# Create baseline score
df["baseline_score"] = (
    (1 - df["ctr"].fillna(0)) * 4 +
    (df["avg_position"].fillna(0) / 50) * 3 +
    (df["trend_pct"].fillna(0).abs() / 100) * 3
)

# Features and target
X = df[["ctr", "avg_position", "trend_pct"]].fillna(0)
y = df["baseline_score"]

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training Samples :", len(X_train))
print("Testing Samples  :", len(X_test))
print("Features Used    :", list(X.columns))

Training Samples : 24000
Testing Samples  : 6000
Features Used    : ['ctr', 'avg_position', 'trend_pct']


## 4. Results (vs baseline)

*# Results (vs Baseline)

The Random Forest model was trained using the same features and target as the baseline.

The model was evaluated using the same train/test split for a fair comparison.

The evaluation uses:

- Mean Absolute Error (MAE)
- R² Score

The results are intended for decision-support only..*

In [6]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import pandas as pd

# Train Model
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

# Predictions
predictions = model.predict(X_test)

# Metrics
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

results = pd.DataFrame({
    "Model": ["Baseline", "Random Forest"],
    "MAE": ["-", round(mae, 4)],
    "R2 Score": ["-", round(r2, 4)]
})

print("Model Comparison")
display(results)

print("\nSample Predictions")
display(pd.DataFrame({
    "Actual": y_test.values[:10],
    "Predicted": predictions[:10]
}))

Model Comparison


,Model,MAE,R2 Score
0,Baseline,-,-
1,Random Forest,0.1793,0.8713



Sample Predictions


,Actual,Predicted
0,5.518,5.51486
1,4.855,4.85912
2,4.000,4.00000
3,6.208,6.21511
4,5.558,5.56767
5,9.006,8.94245
6,3.689,3.68550
7,-93.540,-93.77508
8,7.834,7.83382
9,6.295,6.29755


## 5. Limitations

*# Limitations

This project uses anonymized historical search performance data.

The model identifies observed patterns but cannot prove cause-and-effect relationships.

The recommendations are intended to support editorial decision-making and should not be interpreted as guaranteed ranking improvements.

External factors such as Google's algorithm updates, seasonality, business priorities, and recent content changes are not fully represented in this dataset.*

In [7]:
limitations = [
    "Uses anonymized historical data only.",
    "Cannot prove causal relationships.",
    "Does not capture all external ranking factors.",
    "Recommendations are decision-support only."
]

print("Project Limitations\n")

for i, item in enumerate(limitations, start=1):
    print(f"{i}. {item}")

Project Limitations

1. Uses anonymized historical data only.
2. Cannot prove causal relationships.
3. Does not capture all external ranking factors.
4. Recommendations are decision-support only.


## 6. Ranked recommendations

*# Ranked Recommendations

Based on the baseline score and the Random Forest model, the highest-ranked pages should be reviewed first for content refresh.

Recommended Actions:

- Refresh pages with low CTR and poor average position.
- Review pages with declining trends.
- Monitor pages with stable performance before making changes.
- Prioritize high-impact pages based on the refresh score.

These recommendations are based on observed search performance signals and are intended for decision-support only.The action playbook output — the paper's recommendations section.*

In [8]:
# Top 10 Recommended Pages

recommendations = df.sort_values(
    "baseline_score",
    ascending=False
)[
    [
        "content_id",
        "baseline_score"
    ]
].head(10)

print("Top 10 Refresh Recommendations\n")

display(recommendations)

Top 10 Refresh Recommendations



,content_id,baseline_score
24695,content_dd882c4152ac,1355.530
15405,content_a023517539fe,846.327
14549,content_d020d42e7fcc,788.807
3561,content_22f4d2f58c42,646.786
19697,content_4f1966b37335,516.078
24726,content_aac5bd559d85,480.412
9097,content_32ff84795595,452.062
27305,content_ceedad7ba6dc,349.986
19525,content_f6e1f66b051e,345.910
1132,content_a6c4ef450727,338.719


## 7. Artifacts the paper embeds

*# Artifacts

The following artifacts are included in this project:

- Dataset Summary
- Model Comparison Table
- Sample Predictions
- Feature Importance Table
- Top 10 Ranked Recommendations

These artifacts support the findings presented in the research paper..*

In [9]:
print("Artifacts Generated")

print("- Dataset Summary")
print("- Model Comparison Table")
print("- Sample Predictions")
print("- Feature Importance")
print("- Top 10 Ranked Recommendations")

Artifacts Generated
- Dataset Summary
- Model Comparison Table
- Sample Predictions
- Feature Importance
- Top 10 Ranked Recommendations


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.